# Personal Writing Style LLM - QLoRA Fine-Tuning on Apple Silicon

**Goal:** Fine-tune a small LLM on *your* writing samples (WhatsApp, email, assignments) so it learns your personal style, then generate text that sounds like you wrote it.

**Stack:**
- **Hardware:** MacBook Air M1 (8-16 GB unified memory)
- **Framework:** Apple MLX + mlx-lm (native Apple Silicon, no CUDA needed)
- **Method:** QLoRA - 4-bit quantized base model + low-rank LoRA adapters
- **Base Model:** Qwen2.5-1.5B-Instruct (fits comfortably in 8 GB at 4-bit)
- **Data:** Your pre-parsed writing samples in JSONL format

**Privacy:** Everything runs locally. No data leaves your machine. No API keys needed.

---

## Notebook Sections
1. Environment Setup & Dependency Installation
2. Data Preparation - Build Training JSONL from Your Parsed Text
3. Style Analysis - Understand Your Writing Fingerprint
4. Model Download - Fetch Qwen2.5-1.5B-Instruct in MLX 4-bit Format
5. QLoRA Fine-Tuning - Train Your Personal LoRA Adapter
6. Inference - Generate Text in Your Style
7. Evaluation - Compare Base vs Fine-Tuned Output
8. Export & Serve - Save Adapter for Use in FastAPI Backend


## 1. Environment Setup

We install Apple's MLX ecosystem. `mlx-lm` handles model loading, quantisation,
LoRA training, and generation - all optimised for Apple Silicon unified memory.

> **First time?** Run this cell once. After that you can skip it.


In [10]:
# Install core dependencies 
# mlx     - Apple's array framework (like PyTorch but for Apple Silicon)
# mlx-lm    - High-level LLM utilities: download, quantize, LoRA, generate
# transformers - For tokenizer loading (mlx-lm uses HF tokenizers under the hood)
# huggingface_hub - Model downloads from Hugging Face

!pip install mlx mlx-lm transformers huggingface_hub --quiet

# Verify MLX can see your Apple Silicon GPU
import mlx.core as mx
print(f"MLX backend : {mx.default_device()}")
print(f"MLX version : {mx.__version__}")

import platform
print(f"Architecture : {platform.machine()}") # Should show 'arm64'
print(f"Python    : {platform.python_version()}")



[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
MLX backend  : Device(gpu, 0)
MLX version  : 0.31.1
Architecture : arm64
Python       : 3.12.13


## 2. Project Configuration

Central config so you can tweak everything from one place.


In [11]:
import os
from pathlib import Path

# Paths 
PROJECT_DIR    = Path.home() / "style-llm"
DATA_DIR     = PROJECT_DIR / "data"
RAW_DIR      = DATA_DIR / "raw"      # Your pre-parsed .txt/.jsonl files go here
TRAINING_DIR   = DATA_DIR / "training"   # Generated train/val JSONL
ADAPTER_DIR    = PROJECT_DIR / "adapters"  # Trained LoRA weights saved here
MODEL_CACHE    = PROJECT_DIR / "models"   # Downloaded MLX model cache

# Model Selection 
# Qwen2.5-1.5B-Instruct is the sweet spot for M1 8GB:
# - 4-bit quantized = ~1 GB VRAM
# - Good instruction-following for style mimicry
# - Fast training (~20-40 min for 500 samples)
#
# Alternatives if you have more RAM or want to experiment:
# "mlx-community/SmolLM2-1.7B-Instruct-4bit"  - Smaller, faster, less capable
# "mlx-community/Phi-3.5-mini-instruct-4bit"  - 3.8B params, needs 12+ GB
# "mlx-community/Qwen2.5-3B-Instruct-4bit"   - Better quality, needs 12+ GB

MODEL_NAME = "mlx-community/Qwen2.5-1.5B-Instruct-4bit"

# Training Hyperparameters 
TRAINING_CONFIG = {
  "lora_rank": 16,      # LoRA rank - 8-32 range. 16 is a good default.
                 # Higher = more capacity but more memory & overfitting risk.
  "lora_alpha": 32,      # Scaling factor. Common convention: 2x rank.
  "lora_dropout": 0.05,    # Regularisation. Helps prevent memorisation.
  "lora_target_modules": [  # Which layers to attach LoRA adapters to.
    "self_attn.q_proj",   # Query projection - captures what to attend to
    "self_attn.k_proj",   # Key projection - captures what to match against
    "self_attn.v_proj",   # Value projection - captures what information to extract
    "self_attn.o_proj",   # Output projection - captures how to combine attention
  ],
  "learning_rate": 2e-4,   # AdamW LR. 1e-4 to 5e-4 is the sweet spot for QLoRA.
  "num_epochs": 3,      # 3-5 epochs. Watch val loss - stop if it rises.
  "batch_size": 1,      # Keep at 1-2 for M1 8GB. Increase if you have 16GB.
  "grad_accumulation_steps": 4, # Effective batch size = batch_size × this = 4
  "warmup_ratio": 0.1,    # 10% of steps for LR warmup
  "weight_decay": 0.01,   # L2 regularisation
  "max_seq_length": 512,   # Max tokens per sample. 512 covers most messages/emails.
  "val_split": 0.1,     # 10% held out for validation
  "seed": 42,
}

# Create directories
for d in [RAW_DIR, TRAINING_DIR, ADAPTER_DIR, MODEL_CACHE]:
  d.mkdir(parents=True, exist_ok=True)

print(f"Project root : {PROJECT_DIR}")
print(f"Model     : {MODEL_NAME}")
print(f"LoRA rank   : {TRAINING_CONFIG['lora_rank']}")
print(f"Effective BS : {TRAINING_CONFIG['batch_size'] * TRAINING_CONFIG['grad_accumulation_steps']}")
print("\n Config ready.")


Project root  : /Users/nageshyadav/style-llm
Model         : mlx-community/Qwen2.5-1.5B-Instruct-4bit
LoRA rank     : 16
Effective BS  : 4

✅ Config ready.


## 3. Data Preparation - Build Training JSONL

Your parsers have already extracted your writing samples. This section converts them
into the ChatML format that Qwen expects for fine-tuning.

### Expected Input Format

Place your parsed samples in `~/style-llm/data/raw/` as one or more `.jsonl` files.
Each line should be a JSON object with at least a `"text"` field containing one of your
writing samples (a message, email body, paragraph, etc.).

Optionally include a `"context"` field (what you were replying to, subject line, topic):

```json
{"text": "ah yeah sure, let's do Saturday morning, grab a coffee maybe?", "context": "friend asking about weekend plans"}
{"text": "Please find attached the Q3 report. Let me know if the revenue projections need revision.", "context": "email to manager about quarterly report"}
{"text": "The results demonstrate a statistically significant improvement in F1 score.", "context": "academic assignment on ML evaluation"}
```

> **No context?** That's fine - the pipeline will auto-generate generic prompts.


In [12]:
import json
import random

# Load your parsed writing samples 
def load_raw_samples(raw_dir: Path) -> list[dict]:
  """
  Load all .jsonl files from the raw directory.
  Each line must have at minimum: {"text": "your writing sample"}
  Optionally: {"text": "...", "context": "what prompted this writing"}
  """
  samples = []
  jsonl_files = list(raw_dir.glob("*.jsonl"))
  
  if not jsonl_files:
    print(f" No .jsonl files found in {raw_dir}")
    print(f"  Place your parsed writing samples there and re-run this cell.")
    print(f"  Creating a demo file with example samples for testing...\n")
    create_demo_samples(raw_dir)
    jsonl_files = list(raw_dir.glob("*.jsonl"))
  
  for fpath in jsonl_files:
    with open(fpath, "r", encoding="utf-8") as f:
      for line_num, line in enumerate(f, 1):
        line = line.strip()
        if not line:
          continue
        try:
          obj = json.loads(line)
          if "text" not in obj or len(obj["text"].split()) < 5:
            continue # Skip very short samples (< 5 words)
          samples.append(obj)
        except json.JSONDecodeError:
          print(f"   Skipped malformed JSON at {fpath.name}:{line_num}")
  
  print(f" Loaded {len(samples)} writing samples from {len(jsonl_files)} file(s)")
  return samples


def create_demo_samples(raw_dir: Path):
  """Create a small demo dataset for testing the pipeline."""
  demos = [
    {"text": "ah yeah sure let's grab a coffee Saturday, no rush though", "context": "friend asking about weekend plans"},
    {"text": "sounds good to me, I'll be around the usual spot", "context": "confirming meeting location"},
    {"text": "honestly I think the approach needs rethinking, the current model isn't capturing the edge cases we discussed", "context": "team discussion about ML model performance"},
    {"text": "Please find the updated analysis attached. I've revised the confidence intervals based on your feedback.", "context": "email follow-up on data analysis"},
    {"text": "the key insight here is that the attention mechanism learns positional relationships implicitly, which is why the results improve with longer context windows", "context": "explaining transformer architecture in assignment"},
    {"text": "quick question - did you get a chance to look at the PR I sent yesterday? no pressure, just checking", "context": "Slack message to colleague about code review"},
    {"text": "I'd recommend we go with the simpler architecture first, validate it works end to end, then iterate", "context": "technical decision discussion"},
    {"text": "ha yeah that's exactly what happened to me last time, absolute nightmare", "context": "casual chat about a shared experience"},
    {"text": "The experimental results in Table 3 show a consistent improvement across all metrics, with the most significant gains observed in the low-resource setting", "context": "writing results section of research paper"},
    {"text": "cheers, that's really helpful, I'll update the config and redeploy tonight", "context": "thanking colleague for debugging help"},
  ]
  demo_path = raw_dir / "demo_samples.jsonl"
  with open(demo_path, "w") as f:
    for d in demos:
      f.write(json.dumps(d) + "\n")
  print(f"  Created demo file: {demo_path} ({len(demos)} samples)")


# Load samples
raw_samples = load_raw_samples(RAW_DIR)

# Show a few examples
print("\n Sample Preview ")
for s in raw_samples[:3]:
  ctx = s.get("context", "N/A")
  print(f" Context : {ctx}")
  print(f" Text  : {s['text'][:100]}...")
  print()


📄 Loaded 10 writing samples from 1 file(s)

── Sample Preview ──
  Context : friend asking about weekend plans
  Text    : ah yeah sure let's grab a coffee Saturday, no rush though...

  Context : confirming meeting location
  Text    : sounds good to me, I'll be around the usual spot...

  Context : team discussion about ML model performance
  Text    : honestly I think the approach needs rethinking, the current model isn't capturing the edge cases we ...



## 4. Style Analysis - Your Writing Fingerprint

Before training, let's quantify your writing style. These metrics become part of
the system prompt during training AND inference, giving the model a statistical
anchor for your style even with limited data.


In [13]:
import re
from collections import Counter

def analyse_writing_style(samples: list[dict]) -> dict:
  """
  Extract quantitative style features from writing samples.
  These features are injected into the system prompt during training
  to help the model anchor to your specific patterns.
  """
  all_text = [s["text"] for s in samples]
  
  # Sentence-level metrics 
  all_sentences = []
  for text in all_text:
    sents = re.split(r'[.!?]+', text)
    all_sentences.extend([s.strip() for s in sents if s.strip()])
  
  sent_lengths = [len(s.split()) for s in all_sentences]
  avg_sent_len = sum(sent_lengths) / max(len(sent_lengths), 1)
  
  # Word-level metrics 
  all_words = []
  for text in all_text:
    words = re.findall(r"[a-zA-Z']+", text.lower())
    all_words.extend(words)
  
  word_freq = Counter(all_words)
  vocab_size = len(word_freq)
  total_words = len(all_words)
  
  # Filler words & casual markers 
  fillers = ["like", "just", "actually", "basically", "honestly", "literally",
        "yeah", "ah", "oh", "hmm", "haha", "lol", "sure", "right",
        "anyway", "though", "kinda", "gonna", "wanna", "cheers", "mate"]
  filler_counts = {w: word_freq.get(w, 0) for w in fillers if word_freq.get(w, 0) > 0}
  filler_ratio = sum(filler_counts.values()) / max(total_words, 1)
  
  # Formality score (simple heuristic) 
  formal_markers = ["please", "kindly", "regards", "sincerely", "attached",
           "pursuant", "furthermore", "however", "therefore", "respectively"]
  casual_markers = ["yeah", "lol", "haha", "gonna", "wanna", "nah", "yep",
           "cool", "dude", "cheers", "mate", "tbh", "imo", "btw"]
  
  formal_count = sum(word_freq.get(w, 0) for w in formal_markers)
  casual_count = sum(word_freq.get(w, 0) for w in casual_markers)
  
  if formal_count + casual_count > 0:
    formality = formal_count / (formal_count + casual_count) # 0=casual, 1=formal
  else:
    formality = 0.5 # Neutral
  
  # Punctuation habits 
  all_raw = " ".join(all_text)
  uses_ellipsis = all_raw.count("...") > len(all_text) * 0.1
  uses_exclamation = all_raw.count("!") > len(all_text) * 0.2
  uses_lowercase_start = sum(1 for t in all_text if t[0].islower()) / max(len(all_text), 1)
  avg_msg_length = sum(len(t.split()) for t in all_text) / max(len(all_text), 1)
  
  style = {
    "total_samples": len(samples),
    "total_words": total_words,
    "vocab_size": vocab_size,
    "avg_sentence_length": round(avg_sent_len, 1),
    "avg_message_length": round(avg_msg_length, 1),
    "formality_score": round(formality, 2), # 0=very casual, 1=very formal
    "top_filler_words": dict(sorted(filler_counts.items(), key=lambda x: -x[1])[:5]),
    "filler_word_ratio": round(filler_ratio, 3),
    "starts_lowercase_pct": round(uses_lowercase_start * 100, 1),
    "uses_ellipsis_often": uses_ellipsis,
    "uses_exclamation_often": uses_exclamation,
  }
  return style


# Run style analysis
style_profile = analyse_writing_style(raw_samples)

print(" Your Writing Style Profile \n")
for key, val in style_profile.items():
  print(f" {key:.<35} {val}")

# Classify overall tone
tone = "formal" if style_profile["formality_score"] > 0.6 else \
    "casual" if style_profile["formality_score"] < 0.3 else "mixed"
print(f"\n Overall tone: {tone}")
print(f"\n Style profile extracted. This will be injected into training prompts.")


── Your Writing Style Profile ──

  total_samples...................... 10
  total_words........................ 158
  vocab_size......................... 126
  avg_sentence_length................ 13.2
  avg_message_length................. 15.9
  formality_score.................... 0.4
  top_filler_words................... {'yeah': 2, 'just': 1, 'honestly': 1, 'ah': 1, 'sure': 1}
  filler_word_ratio.................. 0.051
  starts_lowercase_pct............... 70.0
  uses_ellipsis_often................ False
  uses_exclamation_often............. False

  Overall tone: mixed

✅ Style profile extracted. This will be injected into training prompts.


## 5. Build ChatML Training Dataset

Qwen2.5-Instruct uses the **ChatML** format. We convert each sample into a
multi-turn conversation with a style-aware system prompt.

The system prompt includes your extracted style metrics - this is the key trick
that makes fine-tuning work well with small datasets.


In [14]:
def build_system_prompt(style: dict) -> str:
  """
  Construct a system prompt that encodes the user's writing style.
  This prompt is prepended to EVERY training sample so the model
  learns to associate these style descriptors with the target output.
  """
  tone = "formal" if style["formality_score"] > 0.6 else \
      "casual" if style["formality_score"] < 0.3 else "conversational"
  
  fillers = ", ".join(style["top_filler_words"].keys()) if style["top_filler_words"] else "none notable"
  
  lowercase_note = ""
  if style["starts_lowercase_pct"] > 50:
    lowercase_note = "Often starts sentences in lowercase. "
  
  ellipsis_note = "Frequently uses ellipsis (...). " if style["uses_ellipsis_often"] else ""
  excl_note = "Uses exclamation marks liberally. " if style["uses_exclamation_often"] else ""
  
  return (
    f"You are a writing assistant that mimics a specific person's writing style. "
    f"Write EXACTLY as this person would - match their tone, vocabulary, sentence structure, and habits. "
    f"\n\nStyle profile:"
    f"\n- Tone: {tone}"
    f"\n- Average sentence length: {style['avg_sentence_length']} words"
    f"\n- Average message length: {style['avg_message_length']} words"
    f"\n- Common filler words: {fillers}"
    f"\n- Filler word frequency: {style['filler_word_ratio']:.1%} of all words"
    f"\n- {lowercase_note}{ellipsis_note}{excl_note}"
    f"\nDo NOT be more formal or polished than this person. Match their exact register."
  )


def build_training_conversations(
  samples: list[dict],
  style: dict,
  generic_prompts: list[str] = None,
) -> list[dict]:
  """
  Convert raw samples into ChatML training conversations.
  
  Each training example becomes:
   system: <style-aware system prompt>
   user:  <context or generic prompt>
   assistant: <the person's actual writing>
  """
  if generic_prompts is None:
    generic_prompts = [
      "Write a short message in your natural style.",
      "Reply to this in your usual tone.",
      "Write this the way you normally would.",
      "Compose a response in your personal style.",
      "How would you write this?",
      "Draft a quick message.",
      "Write your thoughts on this.",
      "Respond naturally.",
    ]
  
  system_prompt = build_system_prompt(style)
  conversations = []
  
  for sample in samples:
    text = sample["text"]
    context = sample.get("context", None)
    
    # Build the user message
    if context:
      user_msg = f"Context: {context}\nWrite a response in your style."
    else:
      user_msg = random.choice(generic_prompts)
    
    conv = {
      "messages": [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_msg},
        {"role": "assistant", "content": text},
      ]
    }
    conversations.append(conv)
  
  return conversations


# Build and split the dataset 
random.seed(TRAINING_CONFIG["seed"])
conversations = build_training_conversations(raw_samples, style_profile)
random.shuffle(conversations)

# Train / validation split
split_idx = max(1, int(len(conversations) * (1 - TRAINING_CONFIG["val_split"])))
train_data = conversations[:split_idx]
val_data = conversations[split_idx:]

# Write JSONL files 
def write_jsonl(data: list[dict], path: Path):
  with open(path, "w", encoding="utf-8") as f:
    for item in data:
      f.write(json.dumps(item, ensure_ascii=False) + "\n")

train_path = TRAINING_DIR / "train.jsonl"
val_path = TRAINING_DIR / "valid.jsonl"

write_jsonl(train_data, train_path)
write_jsonl(val_data, val_path)

print(f" Dataset Summary:")
print(f"  Total conversations : {len(conversations)}")
print(f"  Training samples  : {len(train_data)}")
print(f"  Validation samples : {len(val_data)}")
print(f"  Train file     : {train_path}")
print(f"  Val file      : {val_path}")

# Preview one training example
print(f"\n Sample Training Conversation ")
preview = train_data[0]["messages"]
for msg in preview:
  role = msg["role"].upper()
  content = msg["content"][:150] + "..." if len(msg["content"]) > 150 else msg["content"]
  print(f" [{role}]: {content}")
print()


📊 Dataset Summary:
   Total conversations : 10
   Training samples    : 9
   Validation samples  : 1
   Train file          : /Users/nageshyadav/style-llm/data/training/train.jsonl
   Val file            : /Users/nageshyadav/style-llm/data/training/valid.jsonl

── Sample Training Conversation ──
  [SYSTEM]: You are a writing assistant that mimics a specific person's writing style. Write EXACTLY as this person would — match their tone, vocabulary, sentence...
  [USER]: Context: casual chat about a shared experience
Write a response in your style.
  [ASSISTANT]: ha yeah that's exactly what happened to me last time, absolute nightmare



## 6. Download the Base Model

Pull the 4-bit quantized Qwen2.5-1.5B-Instruct from Hugging Face.
The MLX community maintains pre-quantized versions - no manual quantisation needed.

This downloads ~1 GB. Cached locally so subsequent runs are instant.


In [15]:
from huggingface_hub import snapshot_download

print(f"⬇ Downloading {MODEL_NAME}...")
print(f"  Cache location: {MODEL_CACHE}\n")

# Download model to local cache
model_path = snapshot_download(
  repo_id=MODEL_NAME,
  local_dir=MODEL_CACHE / MODEL_NAME.split("/")[-1],
  local_dir_use_symlinks=False,
)

print(f"\n Model downloaded to: {model_path}")

# Quick check - list model files
import os
model_files = os.listdir(model_path)
print(f"  Files: {', '.join(sorted(model_files))}")


⬇️  Downloading mlx-community/Qwen2.5-1.5B-Instruct-4bit...
   Cache location: /Users/nageshyadav/style-llm/models



/Users/nageshyadav/projects/write-like-me/lora-ft-venv312/lib/python3.12/site-packages/huggingface_hub/utils/_validators.py:206: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]


✅ Model downloaded to: /Users/nageshyadav/style-llm/models/Qwen2.5-1.5B-Instruct-4bit
   Files: .cache, .gitattributes, README.md, added_tokens.json, config.json, merges.txt, model.safetensors, model.safetensors.index.json, special_tokens_map.json, tokenizer.json, tokenizer_config.json, vocab.json


## 7. QLoRA Fine-Tuning 

This is the core cell. We use `mlx_lm.lora` to run QLoRA training:

- **Base model** stays frozen in 4-bit quantized form (~1 GB memory)
- **LoRA adapters** are the only trainable parameters (~20-40 MB)
- **Effective batch size** = `batch_size × grad_accumulation_steps` = 4

### What QLoRA Does Internally
1. Loads the 4-bit quantized model (NF4 quantisation)
2. Attaches small trainable matrices (rank 16) to attention layers
3. Only backpropagates through these small matrices
4. Result: ~0.5% of parameters are trainable, but captures style patterns effectively

### Expected Training Time on M1 Air
- 100 samples × 3 epochs ≈ 5-10 minutes
- 500 samples × 3 epochs ≈ 20-40 minutes
- 1000 samples × 3 epochs ≈ 45-90 minutes

> **Watch the validation loss.** If it starts rising while training loss keeps falling,
> you're overfitting - reduce epochs or increase LoRA dropout.


In [16]:
import yaml
import subprocess
import time

# Write training config as YAML (mlx-lm reads this) 
lora_config = {
  # LoRA architecture
  "lora_layers": 16,         # Number of transformer layers to apply LoRA to
                     # (from the last layer backwards)
  "lora_parameters": {
    "rank": TRAINING_CONFIG["lora_rank"],
    "alpha": TRAINING_CONFIG["lora_alpha"],
    "dropout": TRAINING_CONFIG["lora_dropout"],
    "scale": TRAINING_CONFIG["lora_alpha"] / TRAINING_CONFIG["lora_rank"], # LoRA scaling
  },
  
  # Training parameters
  "learning_rate": TRAINING_CONFIG["learning_rate"],
  "iters": None, # We'll compute this from epochs
  "batch_size": TRAINING_CONFIG["batch_size"],
  "grad_checkpoint": True,  # Gradient checkpointing - essential for M1 8GB
                # Trades compute for memory by recomputing activations
  
  # Data
  "data": str(TRAINING_DIR),
  "train": True,
  "seed": TRAINING_CONFIG["seed"],
  "max_seq_length": TRAINING_CONFIG["max_seq_length"],
}

# Compute total iterations from epochs
samples_per_epoch = len(train_data)
effective_batch = TRAINING_CONFIG["batch_size"] * TRAINING_CONFIG["grad_accumulation_steps"]
iters_per_epoch = max(1, samples_per_epoch // effective_batch)
total_iters = iters_per_epoch * TRAINING_CONFIG["num_epochs"]
lora_config["iters"] = total_iters

# Validation frequency - check every ~0.5 epoch
val_every = max(1, iters_per_epoch // 2)
save_every = iters_per_epoch # Save checkpoint each epoch

print(" Training Plan ")
print(f"  Samples per epoch   : {samples_per_epoch}")
print(f"  Effective batch size : {effective_batch}")
print(f"  Iterations per epoch : {iters_per_epoch}")
print(f"  Total iterations   : {total_iters}")
print(f"  Validate every    : {val_every} iterations")
print(f"  Save every      : {save_every} iterations")
print(f"  Gradient checkpointing: Enabled (saves memory)")
print()

# Save config
config_path = PROJECT_DIR / "lora_config.yaml"
with open(config_path, "w") as f:
  yaml.dump(lora_config, f, default_flow_style=False)

print(f" Config saved to: {config_path}")
print(f"\n Ready to train. Run the next cell to start fine-tuning.")


── Training Plan ──
   Samples per epoch     : 9
   Effective batch size  : 4
   Iterations per epoch  : 2
   Total iterations      : 6
   Validate every        : 1 iterations
   Save every            : 2 iterations
   Gradient checkpointing: Enabled (saves memory)

📝 Config saved to: /Users/nageshyadav/style-llm/lora_config.yaml

✅ Ready to train. Run the next cell to start fine-tuning.


In [17]:
# Launch QLoRA Training 
# # We use mlx_lm.lora via command line for maximum compatibility and logging.
# This cell will stream training output in real time.
#
# You'll see output like:
# Iter 10: Train loss 2.451, Learning Rate 1.8e-04, Tokens/sec 245.3
# Iter 20: Val loss 2.312
#
# Good signs:
# Train loss decreasing steadily
# Val loss decreasing (or stable)
# Val loss increasing = overfitting → stop training (Ctrl+C)

print(" Starting QLoRA fine-tuning...\n")
print(f"  Model    : {MODEL_NAME}")
print(f"  Adapter out : {ADAPTER_DIR}")
print(f"  Epochs   : {TRAINING_CONFIG['num_epochs']}")
print(f"  Total iters : {total_iters}")
print("=" * 60)

start_time = time.time()

# The mlx-lm lora command handles everything:
# - Loads 4-bit model
# - Attaches LoRA adapters
# - Runs training loop with AdamW
# - Saves adapter weights
!python -m mlx_lm lora \
  --model {MODEL_NAME} \
  --data {str(TRAINING_DIR)} \
  --adapter-path {str(ADAPTER_DIR)} \
  --train \
  --iters {total_iters} \
  --batch-size {TRAINING_CONFIG["batch_size"]} \
  --learning-rate {TRAINING_CONFIG["learning_rate"]} \
  --num-layers 16 \
  --val-batches 2 \
  --steps-per-eval {val_every} \
  --save-every {save_every} \
  --max-seq-length {TRAINING_CONFIG["max_seq_length"]} \
  --grad-checkpoint \
  --seed {TRAINING_CONFIG["seed"]}

elapsed = time.time() - start_time
print(f"\n{'=' * 60}")
print(f"⏱ Training completed in {elapsed/60:.1f} minutes")

# Check adapter size
adapter_files = list(ADAPTER_DIR.glob("*.safetensors"))
if adapter_files:
  total_size = sum(f.stat().st_size for f in adapter_files)
  print(f" Adapter size: {total_size / 1024 / 1024:.1f} MB")
  print(f"  Location: {ADAPTER_DIR}")
  print(f"\n LoRA adapter trained successfully!")
else:
  print("\n No adapter files found. Check the training output above for errors.")


🚀 Starting QLoRA fine-tuning...

   Model       : mlx-community/Qwen2.5-1.5B-Instruct-4bit
   Adapter out : /Users/nageshyadav/style-llm/adapters
   Epochs      : 3
   Total iters : 6
Loading pretrained model
Fetching 9 files: 100%|███████████████████████| 9/9 [00:00<00:00, 148034.26it/s]
Download complete: : 0.00B [00:00, ?B/s]              
Loading datasets
Training
Trainable parameters: 0.342% (5.276M/1543.714M)
Starting training..., iters: 6
Calculating loss...: 100%|████████████████████████| 1/1 [00:00<00:00,  1.28it/s]
Iter 1: Val loss 3.885, Val took 0.783s
Calculating loss...: 100%|████████████████████████| 1/1 [00:00<00:00,  3.08it/s]
Iter 2: Val loss 2.192, Val took 0.329s
Iter 2: Saved adapter weights to /Users/nageshyadav/style-llm/adapters/adapters.safetensors and /Users/nageshyadav/style-llm/adapters/0000002_adapters.safetensors.
Calculating loss...: 100%|████████████████████████| 1/1 [00:00<00:00,  3.01it/s]
Iter 3: Val loss 6.853, Val took 0.336s
Calculating loss...: 10

## 8. Inference - Generate Text in Your Style 

Now the fun part! We load the base model + your trained LoRA adapter and generate
text that mimics your writing style.

The model takes:
- **System prompt:** Your style profile (same one used during training)
- **User prompt:** What to write about
- **Output:** Text that sounds like *you* wrote it


In [ ]:
from mlx_lm import load, generate

# Load base model + LoRA adapter 
print(f"Loading model + adapter...")
print(f"  Base model : {MODEL_NAME}")
print(f"  Adapter  : {ADAPTER_DIR}")

model, tokenizer = load(
  MODEL_NAME,
  adapter_path=str(ADAPTER_DIR),
)

print(" Model loaded with LoRA adapter.\n")

# Build the style-aware system prompt 
system_prompt = build_system_prompt(style_profile)

def generate_in_my_style(
  prompt: str,
  context: str = None,
  max_tokens: int = 256,
  temperature: float = 0.7,
  top_p: float = 0.9,
  repetition_penalty: float = 1.1,
) -> str:
  """
  Generate text in your personal writing style.
  
  Args:
    prompt: What to write about
    context: Optional context (what you're replying to, topic, etc.)
    max_tokens: Maximum length of generation
    temperature: Creativity (0.1=deterministic, 1.0=creative). 0.7 is natural.
    top_p: Nucleus sampling threshold
    repetition_penalty: Penalise repeated phrases (1.0=off, 1.1=mild)
  
  Returns:
    Generated text in your style
  """
  if context:
    user_msg = f"Context: {context}\nWrite a response in your style."
  else:
    user_msg = prompt
  
  # Build ChatML formatted prompt
  messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_msg},
  ]
  
  # Apply chat template (Qwen's ChatML format)
  chat_prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
  )
  
  # Generate
  response = generate(
    model=model,
    tokenizer=tokenizer,
    prompt=chat_prompt,
    max_tokens=max_tokens,
    #temperature=temperature,
    #top_p=top_p,
    #repetition_penalty=repetition_penalty,
    verbose=False,
  )
  
  return response.strip()


print(" Function ready: generate_in_my_style() ")
print("  Use it like: generate_in_my_style('Reply to a friend about dinner plans')")


Loading model + adapter...
   Base model : mlx-community/Qwen2.5-1.5B-Instruct-4bit
   Adapter    : /Users/nageshyadav/style-llm/adapters


Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

✅ Model loaded with LoRA adapter.

── Function ready: generate_in_my_style() ──
   Use it like: generate_in_my_style('Reply to a friend about dinner plans')


In [25]:
from mlx_lm import load, generate
help(generate)

Help on function generate in module mlx_lm.generate:

generate(model: mlx.nn.layers.base.Module, tokenizer: Union[transformers.tokenization_python.PythonBackend, mlx_lm.tokenizer_utils.TokenizerWrapper], prompt: Union[str, List[int]], verbose: bool = False, **kwargs) -> str
    Generate a complete response from the model.

    Args:
       model (nn.Module): The language model.
       tokenizer (PreTrainedTokenizer): The tokenizer.
       prompt (Union[str, List[int]]): The input prompt string or integer tokens.
       verbose (bool): If ``True``, print tokens and timing information.
           Default: ``False``.
       kwargs: The remaining options get passed to :func:`stream_generate`.
          See :func:`stream_generate` for more details.



In [27]:
# Test: Generate samples in your style 

test_prompts = [
  {
    "prompt": "Write a casual message to a friend about meeting up this weekend",
    "context": "friend suggesting Saturday brunch",
  },
  {
    "prompt": "Write a professional email about a project update",
    "context": "updating your manager on ML model improvements",
  },
  {
    "prompt": "Explain a technical concept briefly",
    "context": "colleague asking how LoRA fine-tuning works",
  },
  {
    "prompt": "Write a short reaction to some good news",
    "context": None,
  },
]

print(" Generated Samples (Your Style) \n")
print("=" * 60)

for i, tp in enumerate(test_prompts, 1):
  result = generate_in_my_style(
    prompt=tp["prompt"],
    context=tp.get("context"),
    temperature=0.7,
    max_tokens=150,
  )
  print(f"\n Prompt {i}: {tp['prompt']}")
  if tp.get("context"):
    print(f"  Context: {tp['context']}")
  print(f"  Output : {result}")
  print("-" * 60)


── Generated Samples (Your Style) ──



TypeError: generate_step() got an unexpected keyword argument 'top_p'

## 9. Evaluation - Base Model vs Your Fine-Tuned Model

Let's compare the raw base model output against your fine-tuned version
to see how much style adaptation the LoRA training achieved.


In [ ]:
# Load base model WITHOUT adapter for comparison 
print("Loading base model (no adapter) for comparison...")
base_model, base_tokenizer = load(MODEL_NAME)

def generate_base(prompt: str, context: str = None, max_tokens: int = 150) -> str:
  """Generate from the base model without any fine-tuning."""
  if context:
    user_msg = f"Context: {context}\nWrite a response."
  else:
    user_msg = prompt
  
  messages = [
    {"role": "system", "content": "You are a helpful writing assistant. Write naturally."},
    {"role": "user", "content": user_msg},
  ]
  chat_prompt = base_tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
  )
  return generate(
    model=base_model, tokenizer=base_tokenizer,
    prompt=chat_prompt, max_tokens=max_tokens,
    temp=0.7, verbose=False
  ).strip()


# Side-by-side comparison 
comparison_prompts = [
  "Write a casual reply to a friend asking about your weekend",
  "Write a brief professional update about a data pipeline fix",
]

print(" Side-by-Side Comparison: Base vs Fine-Tuned \n")
print("=" * 70)

for prompt in comparison_prompts:
  base_output = generate_base(prompt)
  tuned_output = generate_in_my_style(prompt, temperature=0.7, max_tokens=150)
  
  print(f"\n Prompt: {prompt}")
  print(f"\n  BASE MODEL:")
  print(f"  {base_output}")
  print(f"\n  YOUR STYLE (Fine-Tuned):")
  print(f"  {tuned_output}")
  print("-" * 70)

print("\n Notice the differences in tone, vocabulary, and sentence structure.")
print("  The fine-tuned model should sound more like *you*.")

# Clean up base model from memory
del base_model, base_tokenizer
import gc; gc.collect()
print("\n Base model unloaded to free memory.")


## 10. Export & Serve - Ready for Your Backend

Your LoRA adapter is saved and ready to be loaded by the FastAPI backend.

The adapter directory contains:
- `adapter_config.json` - LoRA architecture (rank, alpha, target modules)
- `adapters.safetensors` - Trained weights (~20-40 MB)

To use in production: load `MODEL_NAME` + `ADAPTER_DIR` in your FastAPI app.


In [ ]:
import shutil

# Save style profile alongside adapter 
style_path = ADAPTER_DIR / "style_profile.json"
with open(style_path, "w") as f:
  json.dump(style_profile, f, indent=2)

# Save the system prompt for reuse 
prompt_path = ADAPTER_DIR / "system_prompt.txt"
with open(prompt_path, "w") as f:
  f.write(build_system_prompt(style_profile))

# Summary 
print(" Export Summary \n")
print(f" Adapter directory: {ADAPTER_DIR}")
print(f"  Contents:")
for f in sorted(ADAPTER_DIR.iterdir()):
  size = f.stat().st_size
  unit = "KB" if size < 1024*1024 else "MB"
  size_val = size / 1024 if unit == "KB" else size / (1024*1024)
  print(f"   {f.name:.<40} {size_val:.1f} {unit}")

print(f"\n Quick-Start for FastAPI Backend ")
print(f"""
  from mlx_lm import load, generate
  
  model, tokenizer = load(
    "{MODEL_NAME}",
    adapter_path="{ADAPTER_DIR}",
  )
  
  # Then use generate() with the saved system prompt
""")

print("\n All done! Your personal style LLM is trained and ready to serve.")
print("  Total disk usage: base model (~1 GB) + adapter (~30 MB)")
print("  RAM at inference: ~2-3 GB (4-bit model + adapter)")


## 11. Interactive Playground

Run this cell to enter an interactive loop where you can test prompts.
Type `quit` to exit.


In [ ]:
# Interactive generation loop 
print(" Interactive Style Generator")
print("  Type a prompt and get a response in your writing style.")
print("  Type 'quit' to exit.\n")

while True:
  user_input = input("You: ").strip()
  if user_input.lower() in ("quit", "exit", "q"):
    print("\n Goodbye!")
    break
  if not user_input:
    continue
  
  output = generate_in_my_style(
    prompt=user_input,
    temperature=0.7,
    max_tokens=200,
  )
  print(f"\nStyled: {output}\n")
